Below is my cleaning and joining code for my FEMA Disaster Declaration Summary into our Master Dataset. 

In [1]:
library(dplyr)
library(lubridate)
setwd("..")
disasters <- read.csv("mprestopino3/DisasterDeclarationsSummaries.csv")
master_data <- read.csv("master_clean.csv")

disasters$year <- year(disasters$declarationDate)
state_lookup <- data.frame(abb = state.abb, name = state.name)
disasters <- left_join(disasters, state_lookup, by = c("state" = "abb"))
disasters$state <- disasters$name
disasters$name <- NULL

disaster_counts <- disasters %>%
  filter(!is.na(state)) %>% 
  group_by(state, year) %>%
  summarise(disaster_declarations = n())
final_data <- left_join(master_data, disaster_counts, by = c("state", "year"))
final_data$disaster_declarations[is.na(final_data$disaster_declarations)] <- 0

final_data$workers_affected_per_capita <- final_data$workers_affected / final_data$pop
final_data$layoffs_per_capita <- final_data$layoffs / final_data$pop
final_data <- final_data %>%
  arrange(state, year) %>%
  group_by(state) %>%
  mutate(
    layoffs_per_capita_lag1 = lag(layoffs_per_capita, 1),
    layoffs_per_capita_lag2 = lag(layoffs_per_capita, 2),
    disaster_declarations_lag1 = lag(disaster_declarations, 1)
  ) %>%
  ungroup()

cpi_data <- data.frame(
  year = 2010:2025,
  cpi = c(218.056, 224.939, 229.594, 232.957, 236.736, 237.017, 240.007, 245.120, 251.107, 255.657, 258.811, 270.970, 292.655, 304.702, 314.055, 322.531)
)
final_data <- left_join(final_data, cpi_data, by = "year")
cpi_baseline <- cpi_data$cpi[cpi_data$year == 2010]
final_data <- final_data %>%
  mutate(zhvi_real = zhvi_avg * (cpi_baseline / cpi))

head(final_data)
write.csv(final_data, "mprestopino3/master_with_disasters.csv", row.names = FALSE)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union


`summarise()` has grouped output by 'state'. You can override using the
`.groups` argument.


state,year,zhvi_avg,warn_events,workers_affected,closures,layoffs,pop,births,deaths,netmig,disaster_declarations,workers_affected_per_capita,layoffs_per_capita,layoffs_per_capita_lag1,layoffs_per_capita_lag2,disaster_declarations_lag1,cpi,zhvi_real
<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Alabama,2010,125684.4,30,3993,21,9,4785514,NA,NA,NA,3,0.0008343931,1.880676e-06,NA,NA,NA,218.056,125684.4
Alabama,2011,119875.3,36,5445,22,14,4799642,NA,NA,NA,134,0.0011344596,2.916884e-06,1.880676e-06,NA,3,224.939,116207.2
Alabama,2012,121659.0,32,5156,18,14,4816632,NA,NA,NA,11,0.0010704575,2.906595e-06,2.916884e-06,1.880676e-06,134,229.594,115545.1
Alabama,2013,128972.4,41,7329,26,15,4831586,NA,NA,NA,0,0.0015168932,3.104571e-06,2.906595e-06,2.916884e-06,11,232.957,120722.8
Alabama,2014,133206.1,17,4454,9,8,4843737,NA,NA,NA,21,0.0009195380,1.651617e-06,3.104571e-06,2.906595e-06,0,236.736,122695.3
Alabama,2015,135095.1,26,4946,13,13,4854803,NA,NA,NA,0,0.0010187849,2.677761e-06,1.651617e-06,3.104571e-06,21,237.017,124287.7
